In [1]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter
from matplotlib import pyplot as plt

In [2]:
df = pd.read_csv("/kaggle/input/datasets/aviksarkar124/welfake-isot-dateset-combine/MainTwo.csv")

In [3]:
df

,Unnamed: 0,title,text,label
0,0,BREAKING: FELON WEARING BLACK LIVES MATTER T-S...,Obama s war against cops and white people in f...,0
1,1,Eurogroup head Dijsselbloem to leave Dutch pol...,AMSTERDAM (Reuters) - Outgoing Dutch Finance M...,1
2,2,"Trump growing frustrated with China, weighs tr...",WASHINGTON (Reuters) - President Donald Trump ...,1
3,3,Trump Finally Outspends Hillary On TV Spots,"WATCH: UK Reporter RIPS CNN, Calls Them “Clint...",0
4,4,Exclusive: Germany wants more EU sanctions on ...,BRUSSELS (Reuters) - Germany is urging the Eur...,1
...,...,...,...,...
116430,116430,STUNNING GRAPHIC Of Obama’s Legacy That Every ...,The Democrats doubled down on moving to the le...,0
116431,116431,"Trump says open to bilateral Canada, Mexico pa...",WASHINGTON (Reuters) - U.S. President Donald T...,1
116432,116432,"Blast in Turkish textile factory kills five, w...",ANKARA (Reuters) - An explosion at a textile f...,1
116433,116433,"Orlando Gunman Was Shot at Least 8 Times, Auto...",The authorities in Florida said Friday that Om...,1


In [4]:
df['content'] = df['title'] + "" +  df['text']

In [5]:
df = df.drop(columns = ["title","text","Unnamed: 0"])

In [6]:
df

,label,content
0,0,BREAKING: FELON WEARING BLACK LIVES MATTER T-S...
1,1,Eurogroup head Dijsselbloem to leave Dutch pol...
2,1,"Trump growing frustrated with China, weighs tr..."
3,0,Trump Finally Outspends Hillary On TV SpotsWAT...
4,1,Exclusive: Germany wants more EU sanctions on ...
...,...,...
116430,0,STUNNING GRAPHIC Of Obama’s Legacy That Every ...
116431,1,"Trump says open to bilateral Canada, Mexico pa..."
116432,1,"Blast in Turkish textile factory kills five, w..."
116433,1,"Orlando Gunman Was Shot at Least 8 Times, Auto..."


In [7]:
df['tokens'] = df['content'].apply(lambda x: x.split())

all_words = [word for tokens in df['tokens'] for word in tokens]
word_counts = Counter(all_words)

vocab_size = 20000
vocab = ['<PAD>', '<UNK>'] + [word for word, count in word_counts.most_common(vocab_size)]
word2idx = {word: idx for idx, word in enumerate(vocab)}

In [8]:
def encode(tokens):
    return [word2idx.get(word, 1) for word in tokens]

df['encoded'] = df['tokens'].apply(encode)

In [9]:
max_len = 500

def pad_sequence(seq, max_len):
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [0] * (max_len - len(seq))

df['padded'] = df['encoded'].apply(lambda x: pad_sequence(x, max_len))

print(df['padded'][0])
print(len(df['padded'][0]))

[3793, 1, 1, 8847, 1, 1, 1, 1, 2168, 1, 5038, 16312, 800, 1, 1, 248, 975, 1, 1, 14, 294, 85, 3611, 5, 312, 50, 7, 497, 3439, 5558, 14, 6, 4271, 7, 87, 14, 294, 9, 214, 515, 1, 1, 181, 3, 230, 4, 1, 7469, 1218, 60, 586, 3, 24, 445, 26, 1429, 328, 32, 376, 597, 1557, 56, 2045, 774, 4, 6, 1, 2850, 474, 20, 5661, 16, 6, 788, 1138, 16373, 12, 6, 12531, 10516, 341, 1606, 16312, 10911, 800, 230, 1, 691, 43, 161, 3, 30, 3626, 821, 865, 124, 1, 14, 4401, 25, 17, 39, 2153, 6, 555, 5, 1423, 3, 1492, 1, 132, 3020, 5, 4816, 10419, 1, 9, 1, 13, 2, 268, 1, 7, 6, 649, 1703, 2148, 19573, 5, 6197, 1, 29, 3554, 31, 778, 63, 30, 1, 1001, 14, 1253, 27, 301, 26, 20, 184, 384, 411, 32, 1, 1001, 5, 20, 331, 43, 1, 5707, 37, 1377, 32, 10, 138, 2923, 25, 1, 1536, 1, 1, 189, 84, 10, 51, 363, 7, 32, 237, 124, 44, 112, 484, 624, 1457, 5, 138, 331, 112, 624, 8065, 21, 10, 7, 138, 2497, 15, 1001, 39, 78, 1962, 26, 2592, 42, 39, 27, 475, 2007, 11, 2, 1758, 59, 4650, 2168, 4315, 43, 1011, 26, 6, 2295, 1, 12697, 74, 20

In [10]:
print(df['padded'][0])
print(len(df['padded'][0]))

[3793, 1, 1, 8847, 1, 1, 1, 1, 2168, 1, 5038, 16312, 800, 1, 1, 248, 975, 1, 1, 14, 294, 85, 3611, 5, 312, 50, 7, 497, 3439, 5558, 14, 6, 4271, 7, 87, 14, 294, 9, 214, 515, 1, 1, 181, 3, 230, 4, 1, 7469, 1218, 60, 586, 3, 24, 445, 26, 1429, 328, 32, 376, 597, 1557, 56, 2045, 774, 4, 6, 1, 2850, 474, 20, 5661, 16, 6, 788, 1138, 16373, 12, 6, 12531, 10516, 341, 1606, 16312, 10911, 800, 230, 1, 691, 43, 161, 3, 30, 3626, 821, 865, 124, 1, 14, 4401, 25, 17, 39, 2153, 6, 555, 5, 1423, 3, 1492, 1, 132, 3020, 5, 4816, 10419, 1, 9, 1, 13, 2, 268, 1, 7, 6, 649, 1703, 2148, 19573, 5, 6197, 1, 29, 3554, 31, 778, 63, 30, 1, 1001, 14, 1253, 27, 301, 26, 20, 184, 384, 411, 32, 1, 1001, 5, 20, 331, 43, 1, 5707, 37, 1377, 32, 10, 138, 2923, 25, 1, 1536, 1, 1, 189, 84, 10, 51, 363, 7, 32, 237, 124, 44, 112, 484, 624, 1457, 5, 138, 331, 112, 624, 8065, 21, 10, 7, 138, 2497, 15, 1001, 39, 78, 1962, 26, 2592, 42, 39, 27, 475, 2007, 11, 2, 1758, 59, 4650, 2168, 4315, 43, 1011, 26, 6, 2295, 1, 12697, 74, 20

In [11]:
X = np.array(df['padded'].tolist())
y = np.array(df['label'].tolist())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [12]:
class FakeGuardLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        out, (hidden, cell) = self.lstm(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.sigmoid(self.fc(out))
        return out

In [13]:
VOCAB_SIZE = len(word2idx)
EMBED_DIM = 64
HIDDEN_DIM = 128
OUTPUT_DIM = 1

model = FakeGuardLSTM(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM)

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
model = model.to(device)

Using device: cpu
